# Scripts for Preparing Tracking Data for Dataset KITTI

## Outline
0. [Finetune ReID Model (Optional)](#finetune)
1. [Run Tracking Models on Target Videos](#tracking)
2. [Filter Tracking Results](#filter)
3. [Extract ReID Features](#reid)
4. [MOT Person ReID Feature Pairwise Average Distance](#test)

In [14]:
import os
os.environ['PYTHONPATH'] = "..:../../mmdetection/:../../CenterNet/src/:" + \
                           "../../mmtracking/:../../CenterTrack/src/:" + \
                           "../../UMA-MOT/:../../deep-person-reid/"
!echo $PYTHONPATH
import sys
sys.path.insert(0, '..') 

..:../../mmdetection/:../../CenterNet/src/:../../mmtracking/:../../CenterTrack/src/:../../UMA-MOT/:../../deep-person-reid/


## 0. Finetune ReID Model (Optional) <a class="anchor" id="finetune"></a>

In [19]:
train_reid_py = '../videosys/train/reid/train_reid.py'
config_file = '../config/train/reid/kitti_osnet_x1_0_triplet_256x128_amsgrad.yaml'
os.system(' '.join(['python', train_reid_py, '--config-file', config_file]))

Show configuration
adam:
  beta1: 0.9
  beta2: 0.999
cuhk03:
  classic_split: False
  labeled_images: False
  use_metric_cuhk03: False
data:
  combineall: False
  height: 256
  k_tfm: 1
  load_train_targets: False
  norm_mean: [0.485, 0.456, 0.406]
  norm_std: [0.229, 0.224, 0.225]
  root: ../storage/dataset/KITTI/
  save_dir: log/osnet_x1_0_kitti_triplet_mrg005_wt10_wx05
  sources: ['kitti']
  split_id: 0
  targets: ['kitti']
  transforms: ['random_flip']
  type: image
  width: 128
  workers: 4
loss:
  name: triplet
  softmax:
    label_smooth: True
  triplet:
    margin: 0.05
    weight_t: 1.0
    weight_x: 0.5
market1501:
  use_500k_distractors: False
model:
  load_weights: ../storage/models/reid/osnet_x1_0_imagenet.pth
  name: osnet_x1_0
  pretrained: True
  resume: 
mot:
  exclude_sequences: []
  vis_threshold: 0.3
rmsprop:
  alpha: 0.99
sampler:
  num_cams: 1
  num_datasets: 1
  num_instances: 4
  train_sampler: RandomSampler
  train_sampler_t: RandomSampler
sgd:
  dampening: 0.0

/home/darenchao/GitRepo/VideoSys/deep-person-reid/torchreid/metrics/rank.py:12: UserWarning: Cython evaluation (very fast so highly recommended) is unavailable, now use python evaluation.
  'Cython evaluation (very fast so highly recommended) is '
Traceback (most recent call last):
  File "../videosys/train/reid/train_reid.py", line 190, in <module>
    main()
  File "../videosys/train/reid/train_reid.py", line 152, in main
    datamanager = build_datamanager(cfg)
  File "../videosys/train/reid/train_reid.py", line 29, in build_datamanager
    return ImageDataManager(**imagedata_kwargs(cfg))
  File "/home/darenchao/GitRepo/VideoSys/VideoSystem/videosys/train/reid/datamanager.py", line 129, in __init__
    mot_exclude_sequences=mot_exclude_sequences
  File "/home/darenchao/GitRepo/VideoSys/deep-person-reid/torchreid/data/datasets/__init__.py", line 40, in init_image_dataset
    'but expected to be one of {}'.format(name, avai_datasets)
ValueError: Invalid dataset name. Received "kitti",

256

## 1. Run Tracking Models on Target Videos <a class="anchor" id="tracking"></a>

In [3]:
def run_all_methods(video_name, train_test='training'):
    dataset_template = '../../storage/dataset/KITTI/{train_test}/image_02/{video_name}/'
    model_config = '../e2e/configs/mmtracking/detector/faster_rcnn_r50_fpn_one_class.py'
    checkpoint = 'https://download.openmmlab.com/mmtracking/mot/faster_rcnn/faster-rcnn_r50_fpn_4e_mot17-half-64ee2ed4.pth'
    output_template = '../../storage/results/kitti/{train_test}/{video_name}/{det_method}-{method}-person.txt'

    # name, file, detection_method, [params]
    params_for_faster_rcnn = ['--config', model_config, '--checkpoint', checkpoint,]
    method_lists = [
        ('tracktor', 'mmt_tracktor_private.py', 'faster_rcnn', params_for_faster_rcnn),
        # ('sort', 'mmt_sort_private.py', 'faster_rcnn', params_for_faster_rcnn),
        # ('deepsort','mmt_deepsort_private.py', 'faster_rcnn', params_for_faster_rcnn),
        # ('uma', 'uma_private.py', 'faster_rcnn', params_for_faster_rcnn),
        # ('center_track', 'centertrack_private.py', 'center_net', [])
    ]
    for py in method_lists:
        name, pyf, det_method, extra_params = py
        tokens = [
            'python', '../e2e/ingestion_runner.py', 'e2e/configs/tracking/'+pyf,
            '--path', dataset_template.format(train_test=train_test, video_name=video_name),
            '--output', output_template.format(train_test=train_test, video_name=video_name, 
                                               method=name, det_method=det_method),
            *extra_params,
        ]
        command = ' '.join(tokens)
        print('working on method: ', name)
        print('executing command: ', command)
        os.system(command)

In [4]:
video_names = ['0019']
for dn in video_names:
    run_all_methods(dn)

working on method:  tracktor
executing command:  python ../e2e/ingestion_runner.py e2e/configs/tracking/mmt_tracktor_private.py --path ../../storage/dataset/KITTI/training/image_02/0019/ --output ../../storage/results/kitti/training/0019/faster_rcnn-tracktor-person.txt --config ../e2e/configs/mmtracking/detector/faster_rcnn_r50_fpn_one_class.py --checkpoint https://download.openmmlab.com/mmtracking/mot/faster_rcnn/faster-rcnn_r50_fpn_4e_mot17-half-64ee2ed4.pth


2022-07-06 22:10:42,926 - mmtrack - INFO - load reid from: https://download.openmmlab.com/mmtracking/mot/reid/tracktor_reid_r50_iter25245-a452f51f.pth
2022-07-06 22:10:42,926 - mmtrack - INFO - Use load_from_http loader
2022-07-06 22:10:42,987 - mmtrack - WARNING - The model and loaded state dict do not match exactly

missing keys in source state_dict: head.bn.weight, head.bn.bias, head.bn.running_mean, head.bn.running_var, head.classifier.weight, head.classifier.bias

/home/darenchao/GitRepo/VideoSys/mmdetection/mmdet/core/anchor/builder.py:16: UserWarning: ``build_anchor_generator`` would be deprecated soon, please use ``build_prior_generator`` 
  '``build_anchor_generator`` would be deprecated soon, please use '
/home/darenchao/GitRepo/VideoSys/mmdetection/mmdet/core/anchor/anchor_generator.py:323: UserWarning: ``grid_anchors`` would be deprecated soon. Please use ``grid_priors`` 
  warnings.warn('``grid_anchors`` would be deprecated soon. '
/home/darenchao/GitRepo/VideoSys/mmdetect

Use load_from_http loader
begin to process: [image folder], path: [../../storage/dataset/KITTI/training/image_02/0019/]
processing: 100/1059, 9%, time elapsed: 19.12s, fps: 5.23
processing: 200/1059, 19%, time elapsed: 42.43s, fps: 4.71
processing: 300/1059, 28%, time elapsed: 62.65s, fps: 4.79
processing: 400/1059, 38%, time elapsed: 82.89s, fps: 4.83
processing: 500/1059, 47%, time elapsed: 108.69s, fps: 4.60
processing: 600/1059, 57%, time elapsed: 129.38s, fps: 4.64
processing: 700/1059, 66%, time elapsed: 151.00s, fps: 4.64
processing: 800/1059, 76%, time elapsed: 181.47s, fps: 4.41
processing: 900/1059, 85%, time elapsed: 215.62s, fps: 4.17
processing: 1000/1059, 94%, time elapsed: 232.83s, fps: 4.30
done. time elapsed: 239.97s. fps: 4.41


In [5]:
video_names = ['0019', '0022', '0023', '0024', '0025', '0026', '0028']
for dn in video_names:
    run_all_methods(dn, 'testing')

working on method:  tracktor
executing command:  python ../e2e/ingestion_runner.py e2e/configs/tracking/mmt_tracktor_private.py --path ../../storage/dataset/KITTI/testing/image_02/0019/ --output ../../storage/results/kitti/testing/0019/faster_rcnn-tracktor-person.txt --config ../e2e/configs/mmtracking/detector/faster_rcnn_r50_fpn_one_class.py --checkpoint https://download.openmmlab.com/mmtracking/mot/faster_rcnn/faster-rcnn_r50_fpn_4e_mot17-half-64ee2ed4.pth


2022-07-06 22:14:52,648 - mmtrack - INFO - load reid from: https://download.openmmlab.com/mmtracking/mot/reid/tracktor_reid_r50_iter25245-a452f51f.pth
2022-07-06 22:14:52,648 - mmtrack - INFO - Use load_from_http loader
2022-07-06 22:14:52,708 - mmtrack - WARNING - The model and loaded state dict do not match exactly

missing keys in source state_dict: head.bn.weight, head.bn.bias, head.bn.running_mean, head.bn.running_var, head.classifier.weight, head.classifier.bias

/home/darenchao/GitRepo/VideoSys/mmdetection/mmdet/core/anchor/builder.py:16: UserWarning: ``build_anchor_generator`` would be deprecated soon, please use ``build_prior_generator`` 
  '``build_anchor_generator`` would be deprecated soon, please use '
/home/darenchao/GitRepo/VideoSys/mmdetection/mmdet/core/anchor/anchor_generator.py:323: UserWarning: ``grid_anchors`` would be deprecated soon. Please use ``grid_priors`` 
  warnings.warn('``grid_anchors`` would be deprecated soon. '
/home/darenchao/GitRepo/VideoSys/mmdetect

Use load_from_http loader
begin to process: [image folder], path: [../../storage/dataset/KITTI/testing/image_02/0019/]
processing: 100/404, 25%, time elapsed: 20.12s, fps: 4.97
processing: 200/404, 50%, time elapsed: 44.93s, fps: 4.45
processing: 300/404, 74%, time elapsed: 57.98s, fps: 5.17
processing: 400/404, 99%, time elapsed: 76.06s, fps: 5.26
done. time elapsed: 77.26s. fps: 5.23
working on method:  tracktor
executing command:  python ../e2e/ingestion_runner.py e2e/configs/tracking/mmt_tracktor_private.py --path ../../storage/dataset/KITTI/testing/image_02/0022/ --output ../../storage/results/kitti/testing/0022/faster_rcnn-tracktor-person.txt --config ../e2e/configs/mmtracking/detector/faster_rcnn_r50_fpn_one_class.py --checkpoint https://download.openmmlab.com/mmtracking/mot/faster_rcnn/faster-rcnn_r50_fpn_4e_mot17-half-64ee2ed4.pth


2022-07-06 22:16:19,592 - mmtrack - INFO - load reid from: https://download.openmmlab.com/mmtracking/mot/reid/tracktor_reid_r50_iter25245-a452f51f.pth
2022-07-06 22:16:19,592 - mmtrack - INFO - Use load_from_http loader
2022-07-06 22:16:19,651 - mmtrack - WARNING - The model and loaded state dict do not match exactly

missing keys in source state_dict: head.bn.weight, head.bn.bias, head.bn.running_mean, head.bn.running_var, head.classifier.weight, head.classifier.bias

/home/darenchao/GitRepo/VideoSys/mmdetection/mmdet/core/anchor/builder.py:16: UserWarning: ``build_anchor_generator`` would be deprecated soon, please use ``build_prior_generator`` 
  '``build_anchor_generator`` would be deprecated soon, please use '
/home/darenchao/GitRepo/VideoSys/mmdetection/mmdet/core/anchor/anchor_generator.py:323: UserWarning: ``grid_anchors`` would be deprecated soon. Please use ``grid_priors`` 
  warnings.warn('``grid_anchors`` would be deprecated soon. '
/home/darenchao/GitRepo/VideoSys/mmdetect

Use load_from_http loader
begin to process: [image folder], path: [../../storage/dataset/KITTI/testing/image_02/0022/]
processing: 100/436, 23%, time elapsed: 12.70s, fps: 7.87
processing: 200/436, 46%, time elapsed: 27.13s, fps: 7.37
processing: 300/436, 69%, time elapsed: 41.33s, fps: 7.26
processing: 400/436, 92%, time elapsed: 52.99s, fps: 7.55
done. time elapsed: 58.25s. fps: 7.48
working on method:  tracktor
executing command:  python ../e2e/ingestion_runner.py e2e/configs/tracking/mmt_tracktor_private.py --path ../../storage/dataset/KITTI/testing/image_02/0023/ --output ../../storage/results/kitti/testing/0023/faster_rcnn-tracktor-person.txt --config ../e2e/configs/mmtracking/detector/faster_rcnn_r50_fpn_one_class.py --checkpoint https://download.openmmlab.com/mmtracking/mot/faster_rcnn/faster-rcnn_r50_fpn_4e_mot17-half-64ee2ed4.pth


...

Use load_from_http loader
begin to process: [image folder], path: [../../storage/dataset/KITTI/testing/image_02/0025/]
processing: 100/176, 57%, time elapsed: 23.16s, fps: 4.32
done. time elapsed: 34.96s. fps: 5.03
working on method:  tracktor
executing command:  python ../e2e/ingestion_runner.py e2e/configs/tracking/mmt_tracktor_private.py --path ../../storage/dataset/KITTI/testing/image_02/0026/ --output ../../storage/results/kitti/testing/0026/faster_rcnn-tracktor-person.txt --config ../e2e/configs/mmtracking/detector/faster_rcnn_r50_fpn_one_class.py --checkpoint https://download.openmmlab.com/mmtracking/mot/faster_rcnn/faster-rcnn_r50_fpn_4e_mot17-half-64ee2ed4.pth


2022-07-06 22:20:25,636 - mmtrack - INFO - load reid from: https://download.openmmlab.com/mmtracking/mot/reid/tracktor_reid_r50_iter25245-a452f51f.pth
2022-07-06 22:20:25,636 - mmtrack - INFO - Use load_from_http loader
2022-07-06 22:20:25,695 - mmtrack - WARNING - The model and loaded state dict do not match exactly

missing keys in source state_dict: head.bn.weight, head.bn.bias, head.bn.running_mean, head.bn.running_var, head.classifier.weight, head.classifier.bias

/home/darenchao/GitRepo/VideoSys/mmdetection/mmdet/core/anchor/builder.py:16: UserWarning: ``build_anchor_generator`` would be deprecated soon, please use ``build_prior_generator`` 
  '``build_anchor_generator`` would be deprecated soon, please use '
/home/darenchao/GitRepo/VideoSys/mmdetection/mmdet/core/anchor/anchor_generator.py:323: UserWarning: ``grid_anchors`` would be deprecated soon. Please use ``grid_priors`` 
  warnings.warn('``grid_anchors`` would be deprecated soon. '
/home/darenchao/GitRepo/VideoSys/mmdetect

Use load_from_http loader
begin to process: [image folder], path: [../../storage/dataset/KITTI/testing/image_02/0026/]
processing: 100/170, 59%, time elapsed: 22.69s, fps: 4.41
done. time elapsed: 39.29s. fps: 4.33
working on method:  tracktor
executing command:  python ../e2e/ingestion_runner.py e2e/configs/tracking/mmt_tracktor_private.py --path ../../storage/dataset/KITTI/testing/image_02/0028/ --output ../../storage/results/kitti/testing/0028/faster_rcnn-tracktor-person.txt --config ../e2e/configs/mmtracking/detector/faster_rcnn_r50_fpn_one_class.py --checkpoint https://download.openmmlab.com/mmtracking/mot/faster_rcnn/faster-rcnn_r50_fpn_4e_mot17-half-64ee2ed4.pth


2022-07-06 22:21:14,696 - mmtrack - INFO - load reid from: https://download.openmmlab.com/mmtracking/mot/reid/tracktor_reid_r50_iter25245-a452f51f.pth
2022-07-06 22:21:14,696 - mmtrack - INFO - Use load_from_http loader
2022-07-06 22:21:14,755 - mmtrack - WARNING - The model and loaded state dict do not match exactly

missing keys in source state_dict: head.bn.weight, head.bn.bias, head.bn.running_mean, head.bn.running_var, head.classifier.weight, head.classifier.bias

/home/darenchao/GitRepo/VideoSys/mmdetection/mmdet/core/anchor/builder.py:16: UserWarning: ``build_anchor_generator`` would be deprecated soon, please use ``build_prior_generator`` 
  '``build_anchor_generator`` would be deprecated soon, please use '
/home/darenchao/GitRepo/VideoSys/mmdetection/mmdet/core/anchor/anchor_generator.py:323: UserWarning: ``grid_anchors`` would be deprecated soon. Please use ``grid_priors`` 
  warnings.warn('``grid_anchors`` would be deprecated soon. '
/home/darenchao/GitRepo/VideoSys/mmdetect

Use load_from_http loader
begin to process: [image folder], path: [../../storage/dataset/KITTI/testing/image_02/0028/]
processing: 100/175, 57%, time elapsed: 22.91s, fps: 4.36
done. time elapsed: 39.41s. fps: 4.44


## 2. Filter Tracking Results: <a class="anchor" id="filter"></a>
- Filter out short objs.
- Filter out detections on the screen edges.
- Filter out small boxes.

In [2]:
import os 
import cv2
import motmetrics as mm

In [3]:
def preprocess_filter(video_name, method, train_test='training', obj_frame_threshold=20):
    method_result_template = '../../storage/results/kitti/{}/{}/faster_rcnn-{}-person.txt'
    dataset_img_path = '../../storage/dataset/KITTI/{}/image_02/{}/'
    output_template = '../../storage/results/kitti/{}/{}/filtered-tracked/faster_rcnn-{}-person.txt'

    dt_result = mm.io.loadtxt(method_result_template.format(train_test, video_name, method))
    dt_result.reset_index(level=['FrameId', 'Id'], inplace=True)
    
    def _filter_short_objs(dt_result):
        id_length = dt_result.groupby('Id')['FrameId'].count().reset_index(name='Count')
        selected_id = set(id_length.loc[id_length['Count'] >= obj_frame_threshold]['Id'])
        dt_result = dt_result[dt_result['Id'].isin(selected_id)]
        return dt_result

    def _filter_bboxes_on_borders(dt_result):
        ip = dataset_img_path.format(train_test, video_name)
        frame_template = cv2.imread(ip + os.listdir(ip)[0])
        height, width, _ = frame_template.shape
        dt_result = dt_result[
            (dt_result['X'] > 0) & (dt_result['Y'] > 0) &
            (dt_result['X'] + dt_result['Width'] < width) &
            (dt_result['Y'] + dt_result['Height'] < height)]
        return dt_result
    
    def _filter_small_bboxes(dt_result):
        min_width, min_height = 20, 50
        dt_result = dt_result[
            (dt_result['Width'] >= min_width) & (dt_result['Height'] >= min_height)]
        return dt_result

    print(video_name, 'origin feats:', len(dt_result), end=', new feats: ')
    # # 1. filter out short objs.
    dt_result = _filter_short_objs(dt_result)
    # # 2. filter out detections on the screen edges
    # dt_result = _filter_bboxes_on_borders(dt_result)
    # # 3. filter out small boxes
    dt_result = _filter_small_bboxes(dt_result)
    print(len(dt_result), end=', ')
    print('tracks:', len(dt_result.groupby('Id')), ', avg bboxes:', 
          round(len(dt_result)/(len(dt_result.groupby('Id'))+1e-10), 4))
    
    save_tracked_txt(output_template.format(train_test, video_name, method), dt_result)

In [4]:
def save_tracked_txt(path, pf):
    parent_dir = os.path.dirname(path)
    if not os.path.isdir(parent_dir):
        os.makedirs(parent_dir)
    with open(path, 'w') as f:
        for _, row in pf.iterrows():
            f.write('{:.0f},{:.0f},{},{},{},{},{},{:.0f},{:.0f},-1\n'.format(
                row['FrameId'], row['Id'], row['X'], row['Y'], row['Width'], 
                row['Height'],row['Confidence'], row['ClassId'], row['Visibility']
            ))

In [5]:
video_names = ['0019']
for dn in video_names:
    preprocess_filter(dn, 'tracktor')

0019 origin feats: 7721, new feats: 3354, tracks: 78 , avg bboxes: 43.0


In [6]:
video_names = ['0019', '0022', '0023', '0024', '0025', '0026', '0028']
for dn in video_names:
    preprocess_filter(dn, 'tracktor', train_test='testing')

0019 origin feats: 3345, new feats: 700, tracks: 28 , avg bboxes: 25.0
0022 origin feats: 3100, new feats: 897, tracks: 25 , avg bboxes: 35.88
0023 origin feats: 2987, new feats: 739, tracks: 16 , avg bboxes: 46.1875
0024 origin feats: 2951, new feats: 882, tracks: 23 , avg bboxes: 38.3478
0025 origin feats: 1702, new feats: 1020, tracks: 23 , avg bboxes: 44.3478
0026 origin feats: 1596, new feats: 655, tracks: 24 , avg bboxes: 27.2917
0028 origin feats: 1511, new feats: 665, tracks: 21 , avg bboxes: 31.6667


## 3. Extract ReID Features <a class="anchor" id="reid"></a>

In [7]:
def extract_features_for_dataset(video_name, noexc=True, train_test='training', loss_name='softmax'):
    mrg, wt, wx = '005', '10', '05'
    if loss_name == 'triplet':
        loss_str = 'triplet_mrg{}_wt{}_wx{}'.format(mrg, wt, wx)
    else:
        loss_str = 'softmax_rx_noexc'
    data_path_template = '../../storage/dataset/KITTI/{}/image_02/{}/'
    result_template = '../../storage/results/kitti/{}/{}/filtered-tracked/faster_rcnn-{}-person.txt'
    feature_save_template = '../../storage/results/kitti/{}/{}/feats-raw-filtered_%s' + \
                            '/faster_rcnn-{}-person-feat-{}-{}.pkl'
    feature_save_template = feature_save_template % loss_str
    methods = [
        # 'sort', 'deepsort', 
        'tracktor'
    ]
    reid_models = [
        'osnet_x1_0',
        # 'resnet50_fc512'
    ]
    model_file_mapping = {
        'noexc': '../../storage/models/reid/osnet_x1_0_kitti_{}.pth'.format(loss_str),
    }
    model_path_name = 'mot3'
    # gen 
    for method in methods:
        for reid_model in reid_models:
            tokens = [
                'python', '../e2e/ingestion_runner.py', 
                'e2e/configs/tools/gen_track_features_torchreid.py',
                '--data_path', data_path_template.format(train_test, video_name),
                '--result_path', result_template.format(train_test, video_name, method),
                '--feature_save_path', 
                feature_save_template.format(train_test, video_name, method, reid_model, model_path_name),
                '--model_name', reid_model,
                '--model_path', model_file_mapping['noexc'] if noexc else model_file_mapping[video_name]
            ]
            command = ' '.join(tokens)
            print('working on method: ', method)
            print('executing command: ', command)
            os.system(command)    

In [8]:
video_names = ['0019']
for vn in video_names:
    extract_features_for_dataset(vn, loss_name='triplet')

working on method:  tracktor
executing command:  python ../e2e/ingestion_runner.py e2e/configs/tools/gen_track_features_torchreid.py --data_path ../../storage/dataset/KITTI/training/image_02/0019/ --result_path ../../storage/results/kitti/training/0019/filtered-tracked/faster_rcnn-tracktor-person.txt --feature_save_path ../../storage/results/kitti/training/0019/feats-raw-filtered_triplet_mrg005_wt10_wx05/faster_rcnn-tracktor-person-feat-osnet_x1_0-mot3.pkl --model_name osnet_x1_0 --model_path ../../storage/models/reid/osnet_x1_0_kitti_triplet_mrg005_wt10_wx05.pth
Successfully loaded imagenet pretrained weights from "/home/darenchao/.cache/torch/checkpoints/osnet_x1_0_imagenet.pth"
** The following layers are discarded due to unmatched keys or layer size: ['classifier.weight', 'classifier.bias']
Model: osnet_x1_0
- params: 2,193,616
- flops: 978,878,352
Successfully loaded pretrained weights from "../../storage/models/reid/osnet_x1_0_kitti_triplet_mrg005_wt10_wx05.pth"
** The following 

/home/darenchao/GitRepo/VideoSys/deep-person-reid/torchreid/metrics/rank.py:12: UserWarning: Cython evaluation (very fast so highly recommended) is unavailable, now use python evaluation.
  'Cython evaluation (very fast so highly recommended) is '
/home/darenchao/GitRepo/VideoSys/VideoSystem/videosys/ingestion/reid/io.py:37: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  df = pd.DataFrame(data = np.array(self.data), columns = self.head)


In [9]:
video_names = ['0019', '0022', '0023', '0024', '0025', '0026', '0028']
for vn in video_names:
    extract_features_for_dataset(vn, train_test='testing', loss_name='triplet')

working on method:  tracktor
executing command:  python ../e2e/ingestion_runner.py e2e/configs/tools/gen_track_features_torchreid.py --data_path ../../storage/dataset/KITTI/testing/image_02/0019/ --result_path ../../storage/results/kitti/testing/0019/filtered-tracked/faster_rcnn-tracktor-person.txt --feature_save_path ../../storage/results/kitti/testing/0019/feats-raw-filtered_triplet_mrg005_wt10_wx05/faster_rcnn-tracktor-person-feat-osnet_x1_0-mot3.pkl --model_name osnet_x1_0 --model_path ../../storage/models/reid/osnet_x1_0_kitti_triplet_mrg005_wt10_wx05.pth
Successfully loaded imagenet pretrained weights from "/home/darenchao/.cache/torch/checkpoints/osnet_x1_0_imagenet.pth"
** The following layers are discarded due to unmatched keys or layer size: ['classifier.weight', 'classifier.bias']
Model: osnet_x1_0
- params: 2,193,616
- flops: 978,878,352
Successfully loaded pretrained weights from "../../storage/models/reid/osnet_x1_0_kitti_triplet_mrg005_wt10_wx05.pth"
** The following lay

/home/darenchao/GitRepo/VideoSys/deep-person-reid/torchreid/metrics/rank.py:12: UserWarning: Cython evaluation (very fast so highly recommended) is unavailable, now use python evaluation.
  'Cython evaluation (very fast so highly recommended) is '
/home/darenchao/GitRepo/VideoSys/VideoSystem/videosys/ingestion/reid/io.py:37: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  df = pd.DataFrame(data = np.array(self.data), columns = self.head)


working on method:  tracktor
executing command:  python ../e2e/ingestion_runner.py e2e/configs/tools/gen_track_features_torchreid.py --data_path ../../storage/dataset/KITTI/testing/image_02/0022/ --result_path ../../storage/results/kitti/testing/0022/filtered-tracked/faster_rcnn-tracktor-person.txt --feature_save_path ../../storage/results/kitti/testing/0022/feats-raw-filtered_triplet_mrg005_wt10_wx05/faster_rcnn-tracktor-person-feat-osnet_x1_0-mot3.pkl --model_name osnet_x1_0 --model_path ../../storage/models/reid/osnet_x1_0_kitti_triplet_mrg005_wt10_wx05.pth
Successfully loaded imagenet pretrained weights from "/home/darenchao/.cache/torch/checkpoints/osnet_x1_0_imagenet.pth"
** The following layers are discarded due to unmatched keys or layer size: ['classifier.weight', 'classifier.bias']
Model: osnet_x1_0
- params: 2,193,616
- flops: 978,878,352
Successfully loaded pretrained weights from "../../storage/models/reid/osnet_x1_0_kitti_triplet_mrg005_wt10_wx05.pth"
** The following lay

/home/darenchao/GitRepo/VideoSys/deep-person-reid/torchreid/metrics/rank.py:12: UserWarning: Cython evaluation (very fast so highly recommended) is unavailable, now use python evaluation.
  'Cython evaluation (very fast so highly recommended) is '
/home/darenchao/GitRepo/VideoSys/VideoSystem/videosys/ingestion/reid/io.py:37: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  df = pd.DataFrame(data = np.array(self.data), columns = self.head)


working on method:  tracktor
executing command:  python ../e2e/ingestion_runner.py e2e/configs/tools/gen_track_features_torchreid.py --data_path ../../storage/dataset/KITTI/testing/image_02/0023/ --result_path ../../storage/results/kitti/testing/0023/filtered-tracked/faster_rcnn-tracktor-person.txt --feature_save_path ../../storage/results/kitti/testing/0023/feats-raw-filtered_triplet_mrg005_wt10_wx05/faster_rcnn-tracktor-person-feat-osnet_x1_0-mot3.pkl --model_name osnet_x1_0 --model_path ../../storage/models/reid/osnet_x1_0_kitti_triplet_mrg005_wt10_wx05.pth


...

working on method:  tracktor
executing command:  python ../e2e/ingestion_runner.py e2e/configs/tools/gen_track_features_torchreid.py --data_path ../../storage/dataset/KITTI/testing/image_02/0026/ --result_path ../../storage/results/kitti/testing/0026/filtered-tracked/faster_rcnn-tracktor-person.txt --feature_save_path ../../storage/results/kitti/testing/0026/feats-raw-filtered_triplet_mrg005_wt10_wx05/faster_rcnn-tracktor-person-feat-osnet_x1_0-mot3.pkl --model_name osnet_x1_0 --model_path ../../storage/models/reid/osnet_x1_0_kitti_triplet_mrg005_wt10_wx05.pth


/home/darenchao/GitRepo/VideoSys/deep-person-reid/torchreid/metrics/rank.py:12: UserWarning: Cython evaluation (very fast so highly recommended) is unavailable, now use python evaluation.
  'Cython evaluation (very fast so highly recommended) is '
/home/darenchao/GitRepo/VideoSys/VideoSystem/videosys/ingestion/reid/io.py:37: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  df = pd.DataFrame(data = np.array(self.data), columns = self.head)


Successfully loaded imagenet pretrained weights from "/home/darenchao/.cache/torch/checkpoints/osnet_x1_0_imagenet.pth"
** The following layers are discarded due to unmatched keys or layer size: ['classifier.weight', 'classifier.bias']
Model: osnet_x1_0
- params: 2,193,616
- flops: 978,878,352
Successfully loaded pretrained weights from "../../storage/models/reid/osnet_x1_0_kitti_triplet_mrg005_wt10_wx05.pth"
** The following layers are discarded due to unmatched keys or layer size: ['classifier.weight', 'classifier.bias']
begin to process: [image folder], path: [../../storage/dataset/KITTI/testing/image_02/0026/]
processing: 100/170, 59%, time elapsed: 2.44s, fps: 40.95
done. time elapsed: 3.89s. fps: 43.70
working on method:  tracktor
executing command:  python ../e2e/ingestion_runner.py e2e/configs/tools/gen_track_features_torchreid.py --data_path ../../storage/dataset/KITTI/testing/image_02/0028/ --result_path ../../storage/results/kitti/testing/0028/filtered-tracked/faster_rcnn-tr

/home/darenchao/GitRepo/VideoSys/deep-person-reid/torchreid/metrics/rank.py:12: UserWarning: Cython evaluation (very fast so highly recommended) is unavailable, now use python evaluation.
  'Cython evaluation (very fast so highly recommended) is '
/home/darenchao/GitRepo/VideoSys/VideoSystem/videosys/ingestion/reid/io.py:37: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  df = pd.DataFrame(data = np.array(self.data), columns = self.head)


Successfully loaded imagenet pretrained weights from "/home/darenchao/.cache/torch/checkpoints/osnet_x1_0_imagenet.pth"
** The following layers are discarded due to unmatched keys or layer size: ['classifier.weight', 'classifier.bias']
Model: osnet_x1_0
- params: 2,193,616
- flops: 978,878,352
Successfully loaded pretrained weights from "../../storage/models/reid/osnet_x1_0_kitti_triplet_mrg005_wt10_wx05.pth"
** The following layers are discarded due to unmatched keys or layer size: ['classifier.weight', 'classifier.bias']
begin to process: [image folder], path: [../../storage/dataset/KITTI/testing/image_02/0028/]
processing: 100/175, 57%, time elapsed: 2.56s, fps: 39.12
done. time elapsed: 4.14s. fps: 42.22


## 4. MOT Person ReID Feature Pairwise Average Distance <a class="anchor" id="test"></a>

In [10]:
import scripts.kitti.test_mot_person_reid_features_pairwise_average as pairwise_avg

In [11]:
def simple_test(dataset, method, reid_network, select_method, reid_model_pth_name='pretrained',
                train_test='train', loss_name='softmax'):
    first_path = '../../storage/results/kitti/{train_test}/'.format(train_test=train_test)
    if loss_name == 'triplet':
        loss_str = '_triplet_mrg{}_wt{}_wx{}'.format(mrg, wt, wx)
        first_path += '{}/feats-raw-filtered%s/' % loss_str
        frame_path_template = '../../storage/dataset/KITTI/{train_test}/image_02/'.format(
            train_test=train_test) + dataset + '/{:06d}.png'
        feat_template = first_path + 'faster_rcnn-{}-person-feat-{}-{}.pkl'
        result_path = first_path + 'reid-feat-person-filtered/{}-{}-pairwise-{}-{}.txt'
        image_result_path = first_path + 'reid-feat-person-images-filtered/{}-{}-pairwise-{}-{}/'
        dis_path = first_path + 'reid-feat-person-filtered/{}-{}-dis-{}.txt'
    elif loss_name == 'softmax':
        loss_str = '_softmax_rx_noexc'
        first_path += '{}/feats-raw-filtered%s/' % loss_str
        frame_path_template = '../../storage/dataset/KITTI/{train_test}/image_02/'.format(
            train_test=train_test) + dataset + '/{:06d}.png'
        feat_template = first_path + 'faster_rcnn-{}-person-feat-{}-{}.pkl'
        result_path = first_path + 'reid-feat-person-filtered/{}-{}-pairwise-{}-{}.txt'
        image_result_path = first_path + 'reid-feat-person-images-filtered/{}-{}-pairwise-{}-{}/'
        dis_path = first_path + 'reid-feat-person-filtered/{}-{}-dis-{}.txt'
    else:
        assert False, "Unknown loss_name!"
    gt_template = '../../storage/dataset/KITTI/training/label_02/{}.txt'
    method_result_template = '../storage/results/kitti/{}/{}/filtered-tracked/faster_rcnn-{}-person.txt'
    print(loss_str)
    print("dataset: {}".format(dataset))
    print("\tprocessing method [{}] with reid model {}".format(method, reid_network))
    hid_result_tuples_dict, all_track_pair_distance_dict, feat_data = pairwise_avg.get_hid_result(
        feat_template.format(dataset, method, reid_network, reid_model_pth_name), select_method)
    detailed_additional_info_dict = None
    if False and train_test == 'train':  # !!! gt for kitti not supported
        additional_info, format_results, detailed_additional_info_dict = produce_track_distance(
            gt_template.format(dataset),
            method_result_template.format(train_test, dataset, method, reid_network, 
                                          select_method, reid_model_pth_name),
            hid_result_tuples_dict, all_track_pair_distance_dict)
        outputs = format_results_for_output(additional_info, format_results, 10)
        # output
        output_path = result_path.format(dataset, method, reid_network, select_method, reid_model_pth_name)
        parent_dir = os.path.dirname(output_path)
        if not os.path.isdir(parent_dir):
            os.makedirs(parent_dir)
        with open(output_path, 'w') as f:
            f.write('\n'.join(outputs))
    # distance file
    out_dis, num_dis = pairwise_avg.format_results_for_dis(hid_result_tuples_dict, 10)
    print("\tnum_dis:", num_dis)
    output_dis_path = dis_path.format(dataset, method, reid_network, reid_model_pth_name)
    parent_dir = os.path.dirname(output_dis_path)
    if not os.path.isdir(parent_dir):
        os.makedirs(parent_dir)
    with open(output_dis_path, 'w') as f:
        f.write('\n'.join(out_dis))
    # producing images
    print("\tproducing images")
    pairwise_avg.produce_images_for_tracks(hid_result_tuples_dict, 10,
        image_result_path.format(dataset, method, reid_network, select_method, reid_model_pth_name),
        feat_data, frame_path_template, detailed_additional_info_dict, 
        generate_raw_frames=True, train_test=train_test)

In [12]:
method_sel = 'avg'
tracking_model = 'tracktor'  # sort, deepsort, tracktor, uma
video_names = ['0019']
for vn in video_names:
    # simple_test(vn, tracking_model, 'osnet_x1_0', method_sel, 'mot3', train_test='training', loss_name='softmax')
    for mrg, wt, wx in zip(['005'], ['10'], ['05']):
        simple_test(vn, tracking_model, 'osnet_x1_0', method_sel, 'mot3',
                    train_test='training', loss_name='triplet')

_triplet_mrg005_wt10_wx05
dataset: 0019
	processing method [tracktor] with reid model osnet_x1_0
	num_dis: 5343
	producing images


In [13]:
video_names = ['0019', '0022', '0023', '0024', '0025', '0026', '0028']
for vn in video_names:
    for mrg, wt, wx in zip(['005'], ['10'], ['05']):
        simple_test(vn, tracking_model, 'osnet_x1_0', method_sel, 'mot3',
                    train_test='testing', loss_name='triplet')

_triplet_mrg005_wt10_wx05
dataset: 0019
	processing method [tracktor] with reid model osnet_x1_0
	num_dis: 674
	producing images
_triplet_mrg005_wt10_wx05
dataset: 0022
	processing method [tracktor] with reid model osnet_x1_0
	num_dis: 505
	producing images
_triplet_mrg005_wt10_wx05
dataset: 0023
	processing method [tracktor] with reid model osnet_x1_0
	num_dis: 181
	producing images
_triplet_mrg005_wt10_wx05
dataset: 0024
	processing method [tracktor] with reid model osnet_x1_0
	num_dis: 405
	producing images
_triplet_mrg005_wt10_wx05
dataset: 0025
	processing method [tracktor] with reid model osnet_x1_0
	num_dis: 283
	producing images
_triplet_mrg005_wt10_wx05
dataset: 0026
	processing method [tracktor] with reid model osnet_x1_0
	num_dis: 389
	producing images
_triplet_mrg005_wt10_wx05
dataset: 0028
	processing method [tracktor] with reid model osnet_x1_0
	num_dis: 268
	producing images
